# 02. Data Cleaning and EDA

**Project:** Aura (Dormathon 2026, Track 1: Predict Early, Decide Better)

**Where we are:** notebook 01 showed that the raw file is usable. It has an outcome column and its lead times behave sensibly. It also has doubtful rows (road shipments across an ocean) and no cold-chain columns.

**Goal of this notebook:**
1. Show how the raw file is rebuilt into the training file, and prove the rebuild is repeatable.
2. Show the check that stops a bad file from getting through.
3. Explore the training file to see what separates disrupted shipments from safe ones.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pio.renderers.default = "plotly_mimetype+notebook_connected"

# Works whether the notebook is run from the project root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))  # lets us import the build scripts from src/

# Chart colours, kept the same in every chart
BLUE, ORANGE, AQUA, OCHRE = "#2a78d6", "#eb6834", "#1baf7a", "#a07a12"
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"
MODE_COLORS = {"Air": BLUE, "Road": ORANGE, "Rail": OCHRE, "Sea": AQUA}
WEATHER_ORDER = ["Clear", "Rain", "Fog", "Storm", "Hurricane"]


def style(fig, title, height=420):
    """Give every chart the same plain look."""
    fig.update_layout(
        title=dict(text=title, font=dict(size=16, color=INK), x=0, xanchor="left"),
        height=height,
        paper_bgcolor=SURFACE,
        plot_bgcolor=SURFACE,
        font=dict(family="Segoe UI, Arial, sans-serif", size=12, color=MUTED),
        margin=dict(l=64, r=32, t=72, b=56),
        hoverlabel=dict(bgcolor="white", font_color=INK),
    )
    fig.update_xaxes(gridcolor=GRID, linecolor=GRID, zeroline=False)
    fig.update_yaxes(gridcolor=GRID, linecolor=GRID, zeroline=False)
    return fig

## Part A. Cleaning

### 1. Rebuild the file

The cleaning lives in two scripts, so anyone on the team can re-create the files with one command: `python src/build_dataset.py`.

- `src/clean_dataset.py` checks the raw file, fixes the types and adds the quality flags. It never changes a recorded value and never deletes a row.
- `src/build_dataset.py` then adds the simulated cold-chain columns.

Here we run the same code and confirm it gives exactly the two files saved in `data/processed/`. The simulated columns use a fixed random seed, so the result is the same every time.

In [2]:
from build_dataset import CLEAN_PATH, FLAG_COLUMNS, OUT_PATH, RAW_PATH, SIMULATED_COLUMNS, TARGET, build
from clean_dataset import clean as clean_raw_file

raw = pd.read_csv(RAW_PATH)
flagged = pd.read_csv(CLEAN_PATH)   # step 1: recorded columns plus flags
clean = pd.read_csv(OUT_PATH)       # step 2: the training file


def as_saved(df):
    """Dates are written to disk as text, so compare them as text."""
    return df.assign(Date=df["Date"].dt.strftime("%Y-%m-%d")).reset_index(drop=True)


rebuilt_flagged = clean_raw_file(raw)
rebuilt = build(rebuilt_flagged)

for label, fresh, saved in [("Flagged file ", rebuilt_flagged, flagged), ("Training file", rebuilt, clean)]:
    try:
        pd.testing.assert_frame_equal(as_saved(fresh), saved, check_dtype=False)
        print(f"{label}: rebuild matches the saved file exactly: True")
    except AssertionError:
        print(f"{label}: rebuild matches the saved file exactly: False. Re-run python src/build_dataset.py")

print()
print(f"Raw file      : {raw.shape[0]:,} rows, {raw.shape[1]} columns")
print(f"Flagged file  : {flagged.shape[0]:,} rows, {flagged.shape[1]} columns")
print(f"Training file : {clean.shape[0]:,} rows, {clean.shape[1]} columns")
clean.head()

Flagged file : rebuild matches the saved file exactly: True
Training file: rebuild matches the saved file exactly: True

Raw file      : 5,000 rows, 14 columns
Flagged file  : 5,000 rows, 18 columns
Training file : 5,000 rows, 23 columns


,Shipment_ID,Date,Place_of_Origin,Place_of_Destination,Distance_km,Transport_Mode,Lead_Time_Days,Transit_Duration_Hours,Stock_Volume,Geopolitical_Risk_Score,Fuel_Price_Index,Carrier_Reliability_Score,Weather_Condition,Product_Category,Buffer_of_Package,Storage_standard,Cargo_Value_USD,Hospital_Urgency_Tier,Flag_Land_Route_Across_Ocean,Flag_Lead_Time_At_Minimum,Flag_Speed_Impossible,Needs_Review,Disruption_Occurred
0,SC-10000,2025-10-16,Singapore,Los Angeles,5930.83,Rail,41.39,752.8,197.42,5.0,2.43,0.865,Hurricane,Textiles,NaN,Ambient,51298.0,Not Applicable,1,0,0,1,1
1,SC-10001,2024-04-24,Singapore,Shanghai,14285.36,Rail,40.92,464.6,237.24,7.5,2.30,0.592,Storm,Automotive,NaN,Ambient,65954.0,Not Applicable,0,0,0,0,1
2,SC-10002,2024-01-26,Rotterdam,Los Angeles,11113.91,Rail,11.54,229.8,427.42,5.6,1.78,0.673,Rain,Perishables,7.8,-20°C,31033.0,Not Applicable,1,0,0,1,0
3,SC-10003,2024-10-08,Busan,Hamburg,9180.55,Rail,53.13,883.4,170.66,0.8,3.20,0.832,Hurricane,Electronics,NaN,Ambient,1585047.0,Not Applicable,0,0,0,0,1
4,SC-10004,2024-09-07,Busan,Singapore,2762.27,Air,0.50,2.2,434.96,1.9,2.77,0.741,Fog,Perishables,21.8,-20°C,2271196.0,Not Applicable,0,1,0,1,1


### 2. What changed, column by column

For every column in the training file: was it kept, renamed or added, and what share of its values is identical to the raw file.

In [3]:
RENAMED = {"Place_of_Origin": "Origin_Port", "Place_of_Destination": "Destination_Port", "Stock_Volume": "Weight_MT"}

what_happened = {
    "Shipment_ID": "kept",
    "Date": "kept",
    "Place_of_Origin": "renamed from Origin_Port",
    "Place_of_Destination": "renamed from Destination_Port",
    "Distance_km": "kept",
    "Transport_Mode": "kept",
    "Lead_Time_Days": "kept",
    "Transit_Duration_Hours": "simulated: hours travelled so far, 10% to 95% of the lead time",
    "Stock_Volume": "renamed from Weight_MT (metric tonnes)",
    "Geopolitical_Risk_Score": "kept",
    "Fuel_Price_Index": "kept",
    "Carrier_Reliability_Score": "kept",
    "Weather_Condition": "kept",
    "Product_Category": "kept",
    "Buffer_of_Package": "simulated: thermal hold hours left, cold cargo only",
    "Storage_standard": "simulated: from the product type",
    "Cargo_Value_USD": "simulated: weight times a value per kg for the product",
    "Hospital_Urgency_Tier": "simulated: pharmaceuticals only",
    "Flag_Land_Route_Across_Ocean": "flag: road or rail to or from Los Angeles",
    "Flag_Lead_Time_At_Minimum": "flag: lead time on the 0.5-day floor",
    "Flag_Speed_Impossible": "flag: faster than the mode allows",
    "Needs_Review": "flag: 1 if any of the three flags is set",
    "Disruption_Occurred": "kept: the target",
}

pd.set_option("display.max_colwidth", 120)  # show the full sentence
changes = pd.DataFrame({"what_happened": what_happened})
changes["values_same_as_raw"] = [
    f"{(raw[RENAMED.get(col, col)] == clean[col]).mean():.0%}" if RENAMED.get(col, col) in raw.columns else "new column"
    for col in changes.index
]
assert list(changes.index) == list(clean.columns), "The table above is out of step with the file"
changes

,what_happened,values_same_as_raw
Shipment_ID,kept,100%
Date,kept,100%
Place_of_Origin,renamed from Origin_Port,100%
Place_of_Destination,renamed from Destination_Port,100%
Distance_km,kept,100%
Transport_Mode,kept,100%
Lead_Time_Days,kept,100%
Transit_Duration_Hours,"simulated: hours travelled so far, 10% to 95% of the lead time",new column
Stock_Volume,renamed from Weight_MT (metric tonnes),100%
Geopolitical_Risk_Score,kept,100%


**What this tells us:** every recorded column is 100% identical to the raw file. Nothing was rewritten. Three columns were renamed to match the names the app uses, four flags were added, and five columns were simulated.

### 3. The quality flags

Notebook 01 found rows that are physically doubtful. We mark them and keep them, so the decision can be changed later. Do the flagged rows behave differently from the rest?

In [4]:
overall = clean[TARGET].mean()

flag_table = pd.DataFrame([
    {"flag": col, "rows": int(clean[col].sum()), "share_of_file": clean[col].mean(),
     "disruption_rate_if_flagged": clean.loc[clean[col] == 1, TARGET].mean(),
     "disruption_rate_if_not": clean.loc[clean[col] == 0, TARGET].mean()}
    for col in FLAG_COLUMNS + ["Needs_Review"]
]).set_index("flag")
flag_table.style.format({"rows": "{:,}", "share_of_file": "{:.1%}",
                         "disruption_rate_if_flagged": "{:.1%}", "disruption_rate_if_not": "{:.1%}"})

,rows,share_of_file,disruption_rate_if_flagged,disruption_rate_if_not
flag,,,,
Flag_Land_Route_Across_Ocean,560,11.2%,59.1%,61.5%
Flag_Lead_Time_At_Minimum,409,8.2%,49.6%,62.3%
Flag_Speed_Impossible,48,1.0%,37.5%,61.5%
Needs_Review,975,19.5%,55.0%,62.8%


**What this tells us:**

- About 1 row in 5 carries at least one flag. Most are road or rail shipments across an ocean.
- **Rows that cross an ocean by land are disrupted about as often as the rest.** The impossible mode does not distort the outcome, so these rows are safe to train on.
- **Rows on the 0.5-day floor are disrupted less often.** That fits notebook 01: bad weather stretches the lead time, so a very short lead time usually means good weather. The flag is picking up weather, not a data error.
- **Decision:** keep every row for training. The flags stay in the file so the app can mark a shipment as "needs review", and so notebook 05 can check the model on flagged and unflagged rows separately.

### 4. The simulated cold-chain columns

The raw file has no packaging, temperature, value or urgency. Aura needs those to rank shipments and to show money at risk, so `src/build_dataset.py` simulates them. Each rule and its source is written at the top of that script.

One rule matters more than the rest: **the simulated columns never read the outcome.** They are built from the recorded inputs plus seeded randomness.

In [5]:
cold = clean[clean["Storage_standard"] != "Ambient"]
by_air = clean["Transport_Mode"] == "Air"

print(f"Cold-chain shipments : {len(cold):,} of {len(clean):,} ({len(cold) / len(clean):.0%})")
print(f"Ambient shipments    : {len(clean) - len(cold):,}  (buffer left blank on purpose, no urgency tier)")
print(f"Hours travelled never exceed the lead time: {(clean['Transit_Duration_Hours'] <= clean['Lead_Time_Days'] * 24 + 0.06).all()}")
print()
display(pd.crosstab(clean["Product_Category"], clean["Storage_standard"]))
display(pd.crosstab(clean["Product_Category"], clean["Hospital_Urgency_Tier"]))

buffer_table = cold.assign(carried_by=np.where(cold["Transport_Mode"] == "Air", "Air (passive shipper)", "Sea, road, rail (powered reefer)")) \
    .groupby(["carried_by", "Storage_standard"])["Buffer_of_Package"].agg(shipments="size", lowest="min", median="median", highest="max")
display(buffer_table.round(1))

value_table = clean.groupby("Product_Category")["Cargo_Value_USD"].agg(shipments="size", median="median", highest="max")
value_table.style.format({"shipments": "{:,}", "median": "${:,.0f}", "highest": "${:,.0f}"})

Cold-chain shipments : 2,015 of 5,000 (40%)
Ambient shipments    : 2,985  (buffer left blank on purpose, no urgency tier)


Hours travelled never exceed the lead time: True



Storage_standard,-20°C,2°C to 8°C,Ambient,Deep Frozen / Dry Ice
Product_Category,,,,
Automotive,0,0,959,0
Electronics,0,0,1016,0
Perishables,397,612,0,0
Pharmaceuticals,255,665,0,86
Textiles,0,0,1010,0


Hospital_Urgency_Tier,Critical Emergency,Elevated Procedure,Not Applicable,Routine Restock
Product_Category,,,,
Automotive,0,0,959,0
Electronics,0,0,1016,0
Perishables,0,0,1009,0
Pharmaceuticals,95,321,0,590
Textiles,0,0,1010,0


shipments  lowest  median  highest
carried_by                       Storage_standard                                         
Air (passive shipper)            -20°C                        161     7.3    28.0    109.2
                                 2°C to 8°C                   325     0.0    22.1    106.6
                                 Deep Frozen / Dry Ice         20     7.5    22.4     77.0
Sea, road, rail (powered reefer) -20°C                        491     6.0     8.0     10.0
                                 2°C to 8°C                   952     1.0     1.5      2.0
                                 Deep Frozen / Dry Ice         66    72.1    87.0     99.8

,shipments,median,highest
Product_Category,,,
Automotive,959,"$84,610","$10,932,499"
Electronics,"1,016","$3,260,007","$478,256,769"
Perishables,"1,009","$24,874","$5,626,557"
Pharmaceuticals,"1,006","$949,898","$147,905,017"
Textiles,"1,010","$61,594","$8,772,212"


**What this tells us:**

- **2 shipments in 5 are cold-chain**: all pharmaceuticals and all perishables. The other three product types travel at ambient temperature and have no buffer.
- **Hospital urgency exists only for pharmaceuticals.** About 1 in 10 pharma shipments is a Critical Emergency.
- **`Buffer_of_Package` means two different things depending on the mode.** By air it is the hold time left in an insulated box, from a few hours to a few days. By sea, road and rail the cargo sits in a powered refrigerated container, and the buffer is how long it survives if the power fails: one or two hours for chilled cargo, longer for frozen.
- **Cargo values are very large and very uneven.** A median pharma shipment is worth close to $1 million, and one electronics shipment is worth over $400 million. The file records weight in tonnes, so each row is a container load, not a parcel. Money figures later in the project should be read as rough sizes.

Now the proof that these columns cannot leak the answer.

In [6]:
SIMULATED_NUMERIC = ["Transit_Duration_Hours", "Buffer_of_Package", "Cargo_Value_USD"]

leak_check = pd.DataFrame({
    "correlation_with_outcome": {c: clean[c].corr(clean[TARGET]) for c in SIMULATED_NUMERIC},
})
# Hours travelled is a random share of the lead time, so it inherits the lead time's link to the outcome.
# The share itself is the part we simulated.
progress = clean["Transit_Duration_Hours"] / (clean["Lead_Time_Days"] * 24)
leak_check.loc["Share of trip completed (the simulated part of hours travelled)"] = progress.corr(clean[TARGET])
display(leak_check.round(3))

for col in ["Storage_standard", "Hospital_Urgency_Tier"]:
    display(clean.groupby(col)[TARGET].agg(disruption_rate="mean", shipments="size").round(3))

,correlation_with_outcome
Transit_Duration_Hours,0.233
Buffer_of_Package,0.035
Cargo_Value_USD,0.009
Share of trip completed (the simulated part of hours travelled),0.009


,disruption_rate,shipments
Storage_standard,,
-20°C,0.594,652
2°C to 8°C,0.618,1277
Ambient,0.616,2985
Deep Frozen / Dry Ice,0.547,86


,disruption_rate,shipments
Hospital_Urgency_Tier,,
Critical Emergency,0.663,95
Elevated Procedure,0.573,321
Not Applicable,0.613,3994
Routine Restock,0.624,590


**What this tells us:**

- Buffer, cargo value and the share of the trip completed have no link to the outcome, as intended.
- `Transit_Duration_Hours` does show a link, but only because it is a share of `Lead_Time_Days`, which is recorded and does move with the outcome.
- Storage standard and urgency tier show small differences between groups. The small groups (86 deep-frozen shipments, 95 Critical Emergency) are too few for those gaps to mean anything.
- **The consequence, stated plainly:** these columns cannot help predict a disruption. They describe *what is at stake* if one happens: how much the cargo is worth, how urgently it is needed, and how long it stays cold. The model predicts the disruption. These columns decide which disruption to deal with first.

### 5. The check that stops a bad file

`clean_dataset.py` refuses a raw file that breaks a basic rule, and says which rule. Here is a small file with three mistakes: a reliability score above 1, a transport mode that does not exist, and a missing lead time.

In [7]:
broken = raw.head(5).copy()
broken.loc[0, "Carrier_Reliability_Score"] = 1.7      # must be between 0 and 1
broken.loc[1, "Transport_Mode"] = "Ship"              # not a mode in the file
broken.loc[2, "Lead_Time_Days"] = np.nan              # missing

try:
    clean_raw_file(broken)
    print("Accepted (this should not happen)")
except ValueError as problem:
    print("Refused with this message:\n")
    print(str(problem).replace(": ", ":\n", 1).replace("; ", ";\n"))

Refused with this message:

Raw file failed checks:
1 missing or unreadable values;
Transport_Mode has unknown values ['Ship'];
Carrier_Reliability_Score has 1 values outside 0 to 1;
Lead_Time_Days has 1 values outside 0 to 365


**What this tells us:** a bad file is stopped at the door with a message that names each problem, instead of being cleaned quietly and trained on.

This check guards the *training file*. A live shipment typed into the app needs its own guard, and that is the job of `src/preprocessing.py`. It was written for the previous column names (`Planned_Transit_Hours`, `Handover_Count`) and has not been updated for this file yet. Notebook 05 measures which fields the model cannot do without, so that guard knows what to insist on.

## Part B. Exploring the training file

From here on we only use the training file. The question for every chart is the same: **what is different about the shipments that are disrupted?**

### 6. How often are shipments disrupted?

In [8]:
disrupted = clean[clean[TARGET] == 1]

print(f"Disrupted     : {len(disrupted):,} of {len(clean):,} ({overall:.1%})")
print(f"Not disrupted : {len(clean) - len(disrupted):,} ({1 - overall:.1%})")
print()
by_year = clean.assign(year=clean["Date"].str[:4]).groupby("year")[TARGET].agg(disruption_rate="mean", shipments="size")
by_month = clean.assign(month=clean["Date"].str[5:7]).groupby("month")[TARGET].mean()
print(f"Lowest and highest month: {by_month.min():.0%} and {by_month.max():.0%}")
by_year.round(3)

Disrupted     : 3,063 of 5,000 (61.3%)
Not disrupted : 1,937 (38.7%)

Lowest and highest month: 58% and 64%


,disruption_rate,shipments
year,,
2024,0.620,2455
2025,0.606,2545


**What this tells us:**

- **About 6 shipments in 10 are disrupted.** This changes how we judge a model. "Alert on everything" would already be right 61% of the time, so precision of 61% is the floor, not an achievement. A useful model has to do two things: put the surest cases at the top, and pick out the shipments that are *safe*, so a coordinator can stop worrying about them.
- **No season and no trend.** The rate is the same in both years, and every month sits between 58% and 64%. The dispatch date tells us nothing, so it stays out of the model.

### 7. Which kinds of shipment are disrupted more?

In [9]:
category_cols = ["Weather_Condition", "Transport_Mode", "Product_Category", "Storage_standard"]

fig = make_subplots(rows=2, cols=2, subplot_titles=category_cols, vertical_spacing=0.16, horizontal_spacing=0.2)
for i, col in enumerate(category_cols):
    grouped = clean.groupby(col)[TARGET].agg(["mean", "size"]).sort_values("mean")
    r, c = i // 2 + 1, i % 2 + 1
    fig.add_trace(go.Bar(
        x=grouped["mean"], y=grouped.index, orientation="h", marker_color=BLUE,
        text=[f"{m:.0%}  (of {n:,})" for m, n in zip(grouped["mean"], grouped["size"])],
        textposition="outside", textfont=dict(color=INK), cliponaxis=False,
        hovertemplate="%{y}: %{x:.1%}<extra></extra>",
    ), row=r, col=c)
    fig.add_vline(x=overall, line=dict(color=MUTED, width=1, dash="dash"), layer="below", row=r, col=c)
fig.update_annotations(font=dict(size=12, color=INK))
fig.update_xaxes(tickformat=".0%", range=[0, 1.35])
style(fig, f"Disruption rate by category (dashed line = all shipments, {overall:.0%})", height=620)
fig.update_layout(showlegend=False, bargap=0.35, margin=dict(l=150))
fig.show()

**What this tells us:**

- **Weather is the whole story among the categories.** The rate climbs from 37% in clear weather to 80% in a storm and 100% in a hurricane.
- **Transport mode makes no difference.** Air, sea, road and rail all sit at 61%.
- **Product type makes almost no difference.** Pharmaceuticals are disrupted as often as car parts. Textiles are a few points higher, which is small enough to be chance.
- **Storage standard makes no difference either**, as section 4 said it would not.

For Aura this means a cold-chain shipment is not disrupted more often than any other. What makes it different is the cost of a disruption: a delayed box of textiles arrives late, a delayed box of vaccines may arrive spoiled.

### 8. Which numbers move with disruption?

We split four inputs into bands and look at the disruption rate inside each band.

In [10]:
panels = [
    ("Geopolitical_Risk_Score", [-0.1, 2, 4, 6, 8, 10], ["0 to 2", "2 to 4", "4 to 6", "6 to 8", "8 to 10"], "Geopolitical risk score"),
    ("Carrier_Reliability_Score", [0.49, 0.6, 0.7, 0.8, 0.9, 1.0], ["0.5 to 0.6", "0.6 to 0.7", "0.7 to 0.8", "0.8 to 0.9", "0.9 to 1.0"], "Carrier reliability score"),
    ("Lead_Time_Days", [0, 2, 5, 10, 20, 40, 400], ["Under 2", "2 to 5", "5 to 10", "10 to 20", "20 to 40", "Over 40"], "Lead time (days)"),
    ("Distance_km", [0, 3000, 6000, 9000, 12000, 15000], ["Under 3k", "3k to 6k", "6k to 9k", "9k to 12k", "Over 12k"], "Distance (km)"),
]

fig = make_subplots(rows=2, cols=2, subplot_titles=[p[3] for p in panels],
                    vertical_spacing=0.2, horizontal_spacing=0.1)
band_tables = {}
for i, (col, edges, labels, title) in enumerate(panels):
    bands = pd.cut(clean[col], bins=edges, labels=labels)
    grouped = clean.groupby(bands, observed=False)[TARGET].agg(["mean", "size"])
    band_tables[title] = grouped
    fig.add_trace(go.Bar(
        x=grouped.index.astype(str), y=grouped["mean"], marker_color=BLUE,
        text=[f"{m:.0%}" for m in grouped["mean"]], textposition="outside",
        textfont=dict(color=INK), cliponaxis=False, customdata=grouped["size"],
        hovertemplate="%{x}<br>Disruption rate: %{y:.1%}<br>Shipments: %{customdata:,}<extra></extra>",
    ), row=i // 2 + 1, col=i % 2 + 1)
    fig.add_hline(y=overall, line=dict(color=MUTED, width=1, dash="dash"), layer="below", row=i // 2 + 1, col=i % 2 + 1)
fig.update_annotations(font=dict(size=12, color=INK))
fig.update_yaxes(tickformat=".0%", range=[0, 1.15])
fig.update_yaxes(title_text="Disruption rate", col=1)
style(fig, "Disruption rate by band (dashed line = all shipments)", height=640)
fig.update_layout(showlegend=False, bargap=0.3)
fig.show()

pd.concat(band_tables).rename(columns={"mean": "disruption_rate", "size": "shipments"}).round(3)

disruption_rate  shipments
Geopolitical risk score   0 to 2                0.441        987
                          2 to 4                0.556        968
                          4 to 6                0.614       1001
                          6 to 8                0.692       1062
                          8 to 10               0.754        982
Carrier reliability score 0.5 to 0.6            0.639        957
                          0.6 to 0.7            0.675        976
                          0.7 to 0.8            0.620        999
                          0.8 to 0.9            0.570       1047
                          0.9 to 1.0            0.564       1021
Lead time (days)          Under 2               0.505       1214
                          2 to 5                0.576        649
                          5 to 10               0.555        904
                          10 to 20              0.533        923
                          20 to 40              0.700        680
                          Over 40               0.962        630
Distance (km)             Under 3k              0.597        895
                          3k to 6k              0.622       1023
                          6k to 9k              0.619       1007
                          9k to 12k             0.605       1036
                          Over 12k              0.619       1039

**What this tells us:**

- **Geopolitical risk matters steadily.** The rate rises from 44% on the calmest routes to 75% on the riskiest, one clear step per band.
- **Carrier reliability matters a little.** Carriers scoring under 0.7 are disrupted about 65% of the time, and those above 0.8 about 57%.
- **Lead time looks powerful, and that needs care.** Shipments with a lead time over 40 days are disrupted 96% of the time. The next section shows why this is mostly weather seen a second time.
- **Distance does not matter.** Every band sits on the average.

### 9. Why long lead times go with disruption

Notebook 01 found that bad weather stretches the lead time. A hurricane makes a trip about eight times longer. So a long lead time is often a sign of bad weather, and bad weather is what causes the disruption.

To test this we compare disrupted and safe shipments **inside the same weather**, using pace (hours per 1,000 km) so that long and short routes are comparable.

In [11]:
view = clean.copy()
view["pace_h_per_1000km"] = view["Lead_Time_Days"] * 24 / (view["Distance_km"] / 1000)

pace = view.pivot_table(index="Weather_Condition", columns=TARGET, values="pace_h_per_1000km",
                        aggfunc="median").loc[WEATHER_ORDER]
pace.columns = ["not_disrupted", "disrupted"]
pace["difference"] = pace["disrupted"] / pace["not_disrupted"] - 1

fig = go.Figure()
for col, label, color in [("not_disrupted", "Not disrupted", BLUE), ("disrupted", "Disrupted", ORANGE)]:
    fig.add_trace(go.Bar(x=pace.index, y=pace[col], name=label, marker_color=color,
                         text=[("" if pd.isna(v) else f"{v:.0f}") for v in pace[col]], textposition="outside",
                         textfont=dict(color=INK), cliponaxis=False,
                         hovertemplate=label + "<br>%{x}<br>%{y:.1f} hours per 1,000 km<extra></extra>"))
fig.update_yaxes(title_text="Median hours per 1,000 km")
style(fig, "Pace of disrupted and safe shipments, inside each weather", height=440)
fig.update_layout(barmode="group", bargap=0.3,
                  legend=dict(orientation="h", y=-0.18, x=0, font=dict(color=INK)), margin=dict(b=90))
fig.show()

print("Hurricane has no blue bar because no hurricane shipment escaped disruption.")
pace.style.format({"not_disrupted": "{:.1f}", "disrupted": "{:.1f}", "difference": "{:+.0%}"}, na_rep="none")

Hurricane has no blue bar because no hurricane shipment escaped disruption.


,not_disrupted,disrupted,difference
Weather_Condition,,,
Clear,19.1,20.4,+7%
Rain,22.5,24.0,+7%
Fog,27.4,29.6,+8%
Storm,52.0,59.2,+14%
Hurricane,none,153.1,none


**What this tells us:**

- Between one weather and the next, pace changes enormously: about 20 hours per 1,000 km in clear weather and about 150 in a hurricane.
- Inside the same weather, disrupted shipments are only 5% to 15% slower than safe ones.
- So lead time carries a large amount of *weather* and only a little *extra* information. Once the model knows the weather, lead time should add little. Notebook 03 measures exactly how little.
- This has a useful side. If the weather field is ever missing, the pace of the shipment still gives it away.

### 10. How the drivers stack up

Weather and geopolitical risk are the two strongest drivers. Here is the disruption rate for every combination of the two.

In [12]:
view["risk_band"] = pd.cut(view["Geopolitical_Risk_Score"], [-0.1, 2, 4, 6, 8, 10],
                           labels=["0 to 2", "2 to 4", "4 to 6", "6 to 8", "8 to 10"])
grid = view.pivot_table(index="Weather_Condition", columns="risk_band", values=TARGET,
                        aggfunc="mean", observed=False).loc[WEATHER_ORDER]
cell_size = view.pivot_table(index="Weather_Condition", columns="risk_band", values=TARGET,
                             aggfunc="size", observed=False).loc[WEATHER_ORDER]

fig = go.Figure(go.Heatmap(
    z=grid.values, x=[str(c) for c in grid.columns], y=grid.index, zmin=0, zmax=1,
    colorscale=[[0, "#f0efec"], [1, "#c8401f"]],
    text=grid.values, texttemplate="%{text:.0%}", textfont=dict(size=13),
    xgap=3, ygap=3, colorbar=dict(title="Disrupted", thickness=12, tickformat=".0%"),
    customdata=cell_size.values,
    hovertemplate="%{y}, risk %{x}<br>Disrupted: %{z:.1%}<br>Shipments: %{customdata}<extra></extra>",
))
fig.update_xaxes(title_text="Geopolitical risk score")
fig.update_yaxes(autorange="reversed")
style(fig, "Disruption rate for each weather and risk level", height=440)
fig.update_layout(margin=dict(l=110))
fig.show()

print(f"Smallest group in the chart: {int(cell_size.values.min())} shipments")
print(f"Safest corner  (clear weather, risk 0 to 2)  : {grid.loc['Clear', '0 to 2']:.0%} disrupted")
print(f"Riskiest corner without a hurricane (storm, risk 8 to 10): {grid.loc['Storm', '8 to 10']:.0%} disrupted")

Smallest group in the chart: 175 shipments
Safest corner  (clear weather, risk 0 to 2)  : 17% disrupted
Riskiest corner without a hurricane (storm, risk 8 to 10): 95% disrupted


**What this tells us:**

- The two drivers add up. Moving right (riskier route) or down (worse weather) raises the rate, and the effects stack.
- **The safest corner is far below the average.** In clear weather on a calm route, fewer than 1 shipment in 5 is disrupted. These are the shipments a coordinator can leave alone.
- **A storm on a risky route is close to certain**, even without a hurricane.
- No single column gives this picture. It takes at least two together, which is the case for a model over a one-column rule.

### 11. Disruptions that are already certain

Every hurricane shipment in the file was disrupted.

In [13]:
hurricane = clean["Weather_Condition"] == "Hurricane"
rest = clean[~hurricane]

print(f"Hurricane shipments                  : {hurricane.sum():,}, of which {clean.loc[hurricane, TARGET].mean():.0%} are disrupted")
print(f"Share of all disruptions they make up: {hurricane.sum() / len(disrupted):.0%}")
print()
print(f"Shipments left without them          : {len(rest):,}")
print(f"Disruption rate among those          : {rest[TARGET].mean():.1%}")

Hurricane shipments                  : 990, of which 100% are disrupted
Share of all disruptions they make up: 32%

Shipments left without them          : 4,010
Disruption rate among those          : 51.7%


**What this tells us:** about a third of all disruptions can be called by anyone who reads the weather report. They still belong at the top of the coordinator's queue, but they do not prove a model is clever. Take them out and the rest is close to a coin flip, 52% against 48%. That is the harder and more useful job. When we score the model in notebooks 04 and 05, we report how it does on those shipments as well as overall.

### 12. How the columns relate to each other

The same picture as in notebook 01, now with the simulated columns included. The target is the last row.

In [14]:
input_numeric = [
    "Distance_km", "Lead_Time_Days", "Transit_Duration_Hours", "Stock_Volume",
    "Geopolitical_Risk_Score", "Fuel_Price_Index", "Carrier_Reliability_Score",
    "Buffer_of_Package", "Cargo_Value_USD",
]
corr = clean[input_numeric + [TARGET]].corr().round(2)

fig = go.Figure(go.Heatmap(
    z=corr.values, x=corr.columns, y=corr.index, zmin=-1, zmax=1,
    colorscale=[[0, "#2a78d6"], [0.5, "#f0efec"], [1, "#e34948"]],
    text=corr.values, texttemplate="%{text:.2f}", textfont=dict(color=INK, size=10),
    xgap=2, ygap=2, colorbar=dict(title="Correlation", thickness=12),
    hovertemplate="%{y}<br>%{x}<br>Correlation: %{z:.2f}<extra></extra>",
))
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(tickangle=-35)
style(fig, "Correlation between the inputs and the target", height=640)
fig.update_layout(margin=dict(l=190, b=150))
fig.show()

print(f"Link between each input and {TARGET}, strongest first:")
corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).to_frame("correlation")

Link between each input and Disruption_Occurred, strongest first:


,correlation
Lead_Time_Days,0.27
Geopolitical_Risk_Score,0.23
Transit_Duration_Hours,0.23
Carrier_Reliability_Score,-0.08
Buffer_of_Package,0.03
Distance_km,0.01
Stock_Volume,0.01
Cargo_Value_USD,0.01
Fuel_Price_Index,0.00


**What this tells us:**

- Lead time, hours travelled and geopolitical risk are the numeric columns that move with the target. Hours travelled is a share of the lead time, so those two are one signal counted twice.
- `Distance_km`, `Stock_Volume`, `Fuel_Price_Index`, `Buffer_of_Package` and `Cargo_Value_USD` show no link to disruption.
- Weather is text, so it does not appear here, and it is the strongest driver of all (section 7).

### 13. What a disruption puts at risk

The prediction only matters because of what is at stake when a shipment is disrupted. Aura's user looks after cold-chain cargo, so we look at that part of the file.

In [15]:
cold = clean[clean["Storage_standard"] != "Ambient"]
cold_disrupted = cold[cold[TARGET] == 1]
pharma = clean[clean["Product_Category"] == "Pharmaceuticals"]

print(f"Cold-chain shipments                        : {len(cold):,}, of which {len(cold_disrupted):,} were disrupted ({len(cold_disrupted) / len(cold):.1%})")
print(f"Cargo value in all cold-chain shipments     : ${cold['Cargo_Value_USD'].sum():,.0f}")
print(f"Cargo value in the disrupted ones           : ${cold_disrupted['Cargo_Value_USD'].sum():,.0f} ({cold_disrupted['Cargo_Value_USD'].sum() / cold['Cargo_Value_USD'].sum():.0%})")
print(f"Median value of one disrupted cold shipment : ${cold_disrupted['Cargo_Value_USD'].median():,.0f}")
print()

by_tier = pharma.groupby("Hospital_Urgency_Tier").agg(
    shipments=("Shipment_ID", "size"), disrupted=(TARGET, "sum"), disruption_rate=(TARGET, "mean"),
    median_value_usd=("Cargo_Value_USD", "median"))
by_tier["value_in_disrupted_usd"] = pharma[pharma[TARGET] == 1].groupby("Hospital_Urgency_Tier")["Cargo_Value_USD"].sum()
by_tier.loc[["Critical Emergency", "Elevated Procedure", "Routine Restock"]].style.format(
    {"disruption_rate": "{:.1%}", "median_value_usd": "${:,.0f}", "value_in_disrupted_usd": "${:,.0f}",
     "shipments": "{:,}", "disrupted": "{:,}"})

Cold-chain shipments                        : 2,015, of which 1,223 were disrupted (60.7%)
Cargo value in all cold-chain shipments     : $10,221,197,298
Cargo value in the disrupted ones           : $6,125,144,508 (60%)
Median value of one disrupted cold shipment : $366,983



,shipments,disrupted,disruption_rate,median_value_usd,value_in_disrupted_usd
Hospital_Urgency_Tier,,,,,
Critical Emergency,95,63,66.3%,"$27,149,407","$1,894,420,839"
Elevated Procedure,321,184,57.3%,"$1,112,959","$2,123,199,610"
Routine Restock,590,368,62.4%,"$797,410","$1,935,409,141"


**What this tells us:**

- About 1,200 cold-chain shipments were disrupted, carrying billions of dollars of simulated cargo.
- **This is value at risk, not value lost.** A disruption is not a write-off. A delayed shipment in a working refrigerated container may arrive late and intact. How much is truly lost depends on how long the delay is and how much thermal buffer is left, and the file does not record that. Any money figure the app shows needs a stated assumption about what share of a disrupted shipment is lost.
- The urgency tiers are disrupted at similar rates. The gaps between them are within what groups this small produce by chance. Urgency does not predict disruption. It decides whose disruption matters most.

### 14. Columns the model must never see

Only one column describes what happened **after** dispatch: the target. Everything else is known while the shipment is moving.

In [16]:
LABEL_COLUMNS = ["Shipment_ID", "Date", "Place_of_Origin", "Place_of_Destination"]   # names, not causes
RANKING_COLUMNS = ["Cargo_Value_USD", "Hospital_Urgency_Tier", "Buffer_of_Package", "Storage_standard", "Transit_Duration_Hours"]
CANDIDATE_INPUTS = [c for c in clean.columns if c not in LABEL_COLUMNS + RANKING_COLUMNS + [TARGET]]

assert set(RANKING_COLUMNS) == set(SIMULATED_COLUMNS)
print("Target                              :", TARGET)
print("Labels, left out of the model       :", LABEL_COLUMNS)
print("Simulated, used for ranking and money:", RANKING_COLUMNS)
print(f"Candidate model inputs ({len(CANDIDATE_INPUTS)})          :", CANDIDATE_INPUTS)

Target                              : Disruption_Occurred
Labels, left out of the model       : ['Shipment_ID', 'Date', 'Place_of_Origin', 'Place_of_Destination']
Simulated, used for ranking and money: ['Cargo_Value_USD', 'Hospital_Urgency_Tier', 'Buffer_of_Package', 'Storage_standard', 'Transit_Duration_Hours']
Candidate model inputs (13)          : ['Distance_km', 'Transport_Mode', 'Lead_Time_Days', 'Stock_Volume', 'Geopolitical_Risk_Score', 'Fuel_Price_Index', 'Carrier_Reliability_Score', 'Weather_Condition', 'Product_Category', 'Flag_Land_Route_Across_Ocean', 'Flag_Lead_Time_At_Minimum', 'Flag_Speed_Impossible', 'Needs_Review']


## What we found and what happens next

### Findings

| # | Finding | What we do about it |
|---|---|---|
| 1 | The rebuild is repeatable and changes no recorded value | Train on `data/processed/aura_shipments.csv` |
| 2 | About 1 row in 5 is flagged as physically doubtful, and flagged rows behave like the rest | Keep every row, keep the flags |
| 3 | The simulated cold-chain columns have no link to the outcome, by design | Use them to rank shipments and size the money, not to predict |
| 4 | 61% of shipments are disrupted | Do not judge the model on accuracy. Look at how well it ranks and how honest its percentages are |
| 5 | Weather is the strongest driver, then geopolitical risk, then carrier reliability | These three are the core of the model |
| 6 | Lead time looks strong because bad weather stretches it | Turn it into a pace feature in notebook 03 and test what it adds |
| 7 | Transport mode, product type, distance, weight, fuel price and date do not matter | Expect the model to ignore them |
| 8 | Every hurricane shipment is disrupted, a third of all disruptions | Report model results with and without them |
| 9 | A disruption is value at risk, not value lost | State the loss assumption wherever money is shown |

### Next

Notebook 03 turns these findings into model features and tests how much each one helps.

### Limits we state openly

- The file looks generated, so the patterns above are the ones built into it, plus random noise. They show the method works. They do not show how often real shipments are disrupted.
- The outcome says only that a disruption happened, not what kind or how severe.
- The cold-chain columns are simulated by us from stated assumptions. They are not measurements.
- The input guard for live shipments in `src/preprocessing.py` still expects the previous column names and needs updating before the app can use this file.